In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[]
)

model = init_chat_model(
    model="google_genai:gemini-2.5-flash-lite"
)

In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """ give information about the weather """

    return f"The weather is sunny"

# extend a model with a tool
model_with_tool = model.bind_tools([get_weather])

# response = model_with_tool.invoke("What is the weather like in kathmandu?")
# response.text


### Tool Execution Loops

In [4]:
# generate tool calls
messages = [{"role": "user", "content": "Whats the weather like in kathmandu?"}]
ai_response = model_with_tool.invoke(messages)
messages.append(ai_response)


#execute tools
for tool_call in ai_response.tool_calls:
    tool_response = get_weather.invoke(tool_call)
    messages.append(tool_response)

#pass to response 
final_response = model_with_tool.invoke(messages)
print(final_response.text)

The weather in Kathmandu is sunny.


In [5]:
messages

[{'role': 'user', 'content': 'Whats the weather like in kathmandu?'},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Kathmandu"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec249-af02-7171-b44b-fe5d1e80eadb-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Kathmandu'}, 'id': '93c561eb-4aaf-46ae-9466-fc4da8bd6ea5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 16, 'total_tokens': 62, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='The weather is sunny', name='get_weather', tool_call_id='93c561eb-4aaf-46ae-9466-fc4da8bd6ea5')]